### What are we making?

A **Variational Autoencoder** that is trained to detangle semantic embeddings from sentences describing perfume scents.

We need to understand the importance of VAE's: 

### A regular VAE looks like this: 

[encoder]-[bottleneck]-[decoder]


##### A regular VAE reconstructs stuff by going from BIG to SMALL to BIG in order to learn probability distributions so that it can make new things. 

The V in variational Auto Encoder is what means distributon, not a single compressed point representation. A regular autoencoder can only reconstruct old things, not make new ones. 

---

##### Important AHA moment

Creating neural networks which could learn to model probability distributions from their data instead of just learning the precise points for recreation is the core idea that set us up for generative AI as we know it. Being able to sample randomly from a learned probability distribution is what we are doing in VAE's GAN's, world models, Etc when we generate new data. VAE's were one of the first to do this in 2013.  

---

encoders are neural networks. They take an input and output 2 things, a mean and log variance which defines a prob distribution. They start by projecting their input through sequential layers to learn distribution mappings then always end with two linear layers (which return two tensors - mean and logvariance). We use both of these later on in our bottleneck. 

The bottle neck is a function that randomly samples from the learned distribution (logvar and mean), using a reparameterization trick $z=mu+sigma • epsilon$ where epsilon is N(0,1) (standard normal distibution). It does this because you cannot do normal gradient descent over a randomly sampled probability distribution. BUT by using this reparameterization trick we are able to differentiate the sampling part of it! 

1) compress embeddings with 4 sequential layer transformations from 768 (our sentence encoder dims) to 128.

2) projects embeddings through multiple layers and which learn to output probability distribution parameters. The mean and log variance are learned mappings which can be used later down the line in bottleneck portion.

3) Bottleneck randomly samples from what we learned above (mean and log variance) to create new information by calculating z which is the mean + sigma * epsilon (which is a standard normal distribution randomly intiialized.) 

Heres what training looks like which might help place us so far:
```
for data - sample in list of data samples:
    mu, logvar = encode(data)
    
    # 3) we are here -> sigma, epsilon, z = Reparameterization

    x_hat = decode(z)

    # TWO PART LOSS!
    recon_loss = mse(x, x_hat)

    # KL_DIVERGENCE - Kullback-Leibler divergence  - for continuous distributions.
    kl_loss = P(x) • log(p(x) / p(q)) • d(x)

    loss = recon_loss + kl_loss
    loss.backward()
    optimizer.step()
```

4) Decoders are like encoders with their dimensions reveresed. They reconstruct randomly sampled points from our learned probability distribution. In our case, the input dimesions were 768, so our decoder's output is 768. 


5) You learn, you reparameterize, you decode, you calculate loss, you update weights. Thats the whole process. 

## ImportantL: How to test that everything works

1) Is your loss decreasing? 
2) Bypass your encoder and try to sample z directly from a continuous distribution of N(0,1) - if the output looks like it makes sense. stuff is working. The latent space should be structured and smooth, so random points map to meaningful outputs.

3) Map things out in 3 dimensions using umapp and define points. 
3) THe goal is that this thing learned to compress and decompress, meaning it learns to reconstruct well.
4) Images are a really easy way to verify, because you can eye check the output against the expected value. 
5) What i ran into the most while working on my first go around of the perfume engine. The embeddings collapse, meaning - they lose all meaning and the manifold you are creating of scent is just producing identical or nonsensical embeddings. **Test by checking KL loss is nonzero and validating that varying z changes decoder output.**


TC-VAE's come from this paper: Isolating Sources of Disentanglement in Variational Autoencoders

In [ ]:
import torch
import torch.nn as nn
from typing import Tuple

# THese are the num of dimesions coming from our sentence transformers, 
# which first create our embeddings from scent scentences.
INPUT_DIMS = 768
LATENT_DIMS = 32
ENCODER_OUTPUT_DIMENSIONS = 128

class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()

        # Okay your first layer squeezes your data zown. 
        self.layer_1 = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(negative_slope=0.01), # I'm experimenting with leaky relu - this is arbitrary and might suck
            nn.Linear(256, 128),
            nn.LeakyReLU(negative_slope=0.01), # Final layer is leakyrelu 
            # because we want to still preserve direction of vectors.
        )

        # Your final connected layers output both log variance 
        # and the mean which define your probability distribution 
        # in latent space of the embeddings passed in.
        self.fc_mu = nn.Linear(ENCODER_OUTPUT_DIMENSIONS, latent_dim)
        self.fc_logvar = nn.Linear(ENCODER_OUTPUT_DIMENSIONS, latent_dim)

    def forward(self, x) -> Tuple[torch.Tensor, torch.Tensor]:
        # x is our input embeddings!
        # Extract features first.
        features = self.layer_1(x)

        mean = self.fc_mu(features)
        logvar = self.fc_logvar(features)
        
        return (mean, logvar)

## Next is building our decoder. 

Our decoder takes in a variable $z$, which is 32 dimesions (thats the shape of our learned embeddings). So the first layer in layer_1 of our Decoder should start with 32, then get bigger from there.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIMS, output_dim=INPUT_DIMS):
        super().__init__()

        self.layer_1 = nn.Sequential(
            nn.Linear(LATENT_DIMS, 128),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Linear(128, 256),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Linear(256, INPUT_DIMS),
        )

    def forward(self, z) -> torch.Tensor:
        reconstructed_output = self.layer_1(z)
        return reconstructed_output


## Now the full training loop.

Going to set up for training. That starts with getting our dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2') # 768

def encode_sentences(sentence:str) -> np.array:
    return model.encode(sentence)


In [ ]:

data_entry = Path('~/Downloads/parfumo_datos.csv')
df = pd.read_csv(data_entry)

In [ ]:
columns = ['Main Accords', 'Top Notes', 'Middle Notes', 'Base Notes', 'Name']

data = df.copy().iloc[:][columns].dropna()

In [ ]:
scentences = []
idx_to_scent = {}

for index, row in data.iterrows():
    scentence = f'''This perfume has Top Notes of: 
        {row['Top Notes']}; 
        Main Accords of: 
        {row['Main Accords']};
        Base Notes of:
        {row['Base Notes']};
        Middle Notes of:
        {row['Middle Notes']};
    '''
    scentences.append(scentence)
    idx_to_scent[index] = row['Name']

    if len(scentences) > 100:
        break

embeddings = model.encode(scentences)

save_path = Path(__name__).parent / 'data'

embed_path = save_path / 'scentences.csv'
idx_path = save_path / 'idx_to_scent.csv'

idx_df = pd.DataFrame.from_dict(idx_to_scent, orient='index', columns=['name'])
idx_df.to_csv(idx_path)

scent_df = pd.DataFrame(embeddings)
scent_df.to_csv(embed_path)


In [ ]:
from sklearn.model_selection import train_test_split
import torch.nn.functional as F


class VAE(nn.Module):
    def __init__(self, input_dims=768, latent_dims=32):
        super().__init__()
        
        self.encoder = Encoder(input_dims, latent_dims)
        self.decoder = Decoder(latent_dims, input_dims)
    
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterization(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar

    def reparameterization(self, mu, logvar):
        # 1. Convert log-variance to standard deviation: std = exp(0.5 * logvar)
        std = torch.exp(0.5 * logvar)
        
        # 2. Sample standard normal random noise (epsilon) matching the exact shape
        eps = torch.randn_like(std)

        z = mu + std * eps
        return z

vae = VAE()
# Train with momentum! torch.optim.Adam, is a little vanilla
# For funzies I'd like to try using a diff optimizer here. 
# could be more interesting, recently read a paper about this:
# https://patricknicolas.substack.com/p/hands-on-stochastic-gradient-langevin
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

epochs = 15

def load_dataset(data_dir='data'):
    data_path = Path(data_dir)
    embeddings_df = pd.read_csv(data_path / 'scentences.csv', index_col=0)
    idx_to_scent = pd.read_csv(data_path / 'idx_to_scent.csv', index_col=0)
    embeddings = torch.tensor(embeddings_df.values, dtype=torch.float32)
    return embeddings, idx_to_scent

embeddings, idx_to_scent = load_dataset()
train_set, test_set = train_test_split(embeddings, test_size=0.2)

for epoch in range(epochs):
    for index, x in enumerate(train_set):
        optimizer.zero_grad()

        x_hat, mu, logvar = vae(x)
        
        recon_loss = F.mse_loss(x_hat, x)
        kl_divergence = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + kl_divergence
        
        loss.backward()
        optimizer.step()

        if index % 10 == 0:
            print(f'Epoch {epoch} | Step {index} | Loss {loss.item():.4f}')
            print(kl_divergence, 'kl divergence')

torch.save(vae.state_dict(), 'data/vae_weights.pt')
    

I noticed that while training my KL divergence got really close to zero - which is making me suspicious that my latent space is collapsing, and we are not capturing meaningful distinctions between perfumes:
```
Epoch 0 | Step 60 | Loss 0.0008
tensor(3.8952e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 70 | Loss 0.0008
tensor(1.9282e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 0 | Loss 0.0007
tensor(2.5719e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 10 | Loss 0.0006
tensor(0.0001, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 20 | Loss 0.0005
tensor(5.3197e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 30 | Loss 0.0006
tensor(2.2173e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 40 | Loss 0.0004
...
Epoch 14 | Step 60 | Loss 0.0003
tensor(0.0001, grad_fn=<MulBackward0>) kl divergence
Epoch 14 | Step 70 | Loss 0.0002
tensor(0.0001, grad_fn=<MulBackward0>) kl divergence
```
To double check, i'm going to revisit the earlier test i mentioned which is to take our trained vae and see what happens when i vary $z$.


### update, my bad.

I noticed that we were summing our kl divergence, which was only summing over 32 latent factors while our other loss function was computing a mean over 768 dimensions. So i swapped the bottom to mean instead of sum and that seems to have fixed it. 

```
Epoch 0 | Step 0 | Loss 0.0109
tensor(0.0018, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 10 | Loss 0.0051
tensor(0.0002, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 20 | Loss 0.0019
tensor(6.8042e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 30 | Loss 0.0014
tensor(2.4436e-05, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 40 | Loss 0.0013
tensor(8.8224e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 50 | Loss 0.0011
tensor(5.3132e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 60 | Loss 0.0007
tensor(2.0023e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 0 | Step 70 | Loss 0.0011
tensor(2.5388e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 0 | Loss 0.0007
tensor(5.4259e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 10 | Loss 0.0005
tensor(8.0122e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 20 | Loss 0.0006
tensor(1.5404e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 30 | Loss 0.0004
tensor(2.1532e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 1 | Step 40 | Loss 0.0004
...
Epoch 14 | Step 60 | Loss 0.0001
tensor(2.4838e-06, grad_fn=<MulBackward0>) kl divergence
Epoch 14 | Step 70 | Loss 0.0002
tensor(3.6880e-07, grad_fn=<MulBackward0>) kl divergence
```

In [ ]:
import umap
import plotly.express as px

vae.eval()
with torch.no_grad():
    mu, logvar = vae.encoder(embeddings)
    latent_vectors = mu.numpy()

reducer = umap.UMAP(n_components=3, random_state=42)
coords_3d = reducer.fit_transform(latent_vectors)

names = idx_to_scent['name'].values

fig = px.scatter_3d(
    x=coords_3d[:, 0],
    y=coords_3d[:, 1],
    z=coords_3d[:, 2],
    hover_name=names,
    title='VAE Latent Space — Perfume Scent Embeddings',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'},
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    scene=dict(
        xaxis=dict(title='', showticklabels=False),
        yaxis=dict(title='', showticklabels=False),
        zaxis=dict(title='', showticklabels=False),
    ),
)
fig.show()

Now we should show what these look like against non reconstructed embeddings, just the ones that come from our sentence transformer. 

### But Michael, thats not a TCVAE - shouldn't we test both VAE against TCVAE and map them with UMAP to visualize the differences between how their embeddings stack up for multiple types of perfumes?

Yes, omnisicient reader, we should. Because it's 12:14am and i have nothing better to do. 

To rehash:

TC-VAE is when a VAE penalizes total correlation with a new variable called Beta or $B_{TC}$ for you math nerds. This acts as a penalty for when we correlate latent dimensions together. It is a way to improve the separation in our model's latent embeddings and since we will ulimately using these to navigate around, it makes sense that they should be distinct (or disentangled). The goal is that each of our learned dimesions represents one factor. Like sweet, or corn, or boozy. Otherwise, multiple dimesions might redundantly encode the same thing. It is an approach for improving the accuracy of our latent space. It does this by splittling the KL divergence term (or calculation) into three parts.